In [3]:
%pip install -q ollama python-dotenv pillow

Note: you may need to restart the kernel to use updated packages.


In [4]:
# %% Import libraries
import ollama
from pathlib import Path
from IPython.display import display, Markdown
import time

In [5]:
# %% Initialize model 
MODEL_NAME = "llava:7b"  

In [6]:
# %% Define image processing function
def process_images(image_paths, prompt, max_retries=3):
    responses = {}
    
    for img_path in image_paths:
        if not Path(img_path).exists():
            print(f"Image not found: {img_path}")
            continue
            
        retries = 0
        while retries < max_retries:
            try:
                response = ollama.generate(
                    model=MODEL_NAME,
                    prompt=prompt,
                    images=[img_path],
                    stream=False
                )
                responses[str(img_path)] = response['response']
                break
            except Exception as e:
                print(f"Error processing {img_path}: {str(e)}")
                retries += 1
                time.sleep(2 ** retries)  # Exponential backoff
                
    return responses

In [7]:
# %% Story generation function with markdown display
def generate_story(prompt, image_paths):
    results = process_images(image_paths, prompt)
    
    for img_path, response in results.items():
        display(Markdown(f"**Image:** `{img_path}`"))
        display(Markdown(f"**Prompt:** {prompt}"))
        display(Markdown(f"**Response:**\n{response}"))
        display(Markdown("---"))


In [13]:
# Single image example
generate_story(
    prompt="Generate a fictional narrative based on the character depicted in the image. The fictional character should have a name and can take the role of any character within a story, such as protagonist, villain, foil, anti-hero, etc. Explore their desires and motives, and frame your story excerpt as an exposition introducing this character",
    image_paths=["images/yiyi-dark.jpg"]
)

KeyboardInterrupt: 